# Audio Effects: Equalizer, Convolution Reverb and Saturation
This notebook covers three of the projects in this folder's README:
1. **Equalizer**: low shelf, peaking and high shelf filters from the Audio EQ Cookbook, in series.
2. **Convolution reverb**: a synthetic room impulse response, its reverb time, and wet/dry mixing.
3. **Saturation and oversampling**: why nonlinear effects alias, and how much oversampling helps.

It builds on biquads and the bilinear transform (`03_Digital_Filters/filter_design_foundations.ipynb`), convolution (`01_Discrete_Time_Signals_and_Systems`) and resampling (`03_Digital_Filters/resampling.ipynb`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython import display
import ipywidgets as widgets
from ipywidgets import interactive
from audiodsp import rbj_biquad, time_axis

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

FS = 44100
rng = np.random.default_rng(0)

def pluck(frequency, duration, fs=FS):
    """A plucked note: 8 harmonics at 1/k amplitude, fast attack, exponential decay."""
    t = time_axis(duration, fs)
    note = sum(np.sin(2 * np.pi * k * frequency * t) / k for k in range(1, 9) if k * frequency < fs / 2)
    return note * (1 - np.exp(-t / 0.003)) * np.exp(-t / 0.3)

def hat(duration, fs=FS):
    """A hi-hat-like noise burst."""
    t = time_axis(duration, fs)
    return rng.normal(0, 1, len(t)) * np.exp(-t / 0.015)

# Two seconds of test music: a bass and melody line plus hi-hats, with content across the whole spectrum
music = np.zeros(2 * FS)
for i, (bass, lead) in enumerate([(55, 440), (55, 660), (82.4, 550), (82.4, 880),
                                  (73.4, 587), (73.4, 740), (55, 440), (55, 1320)]):
    start = int(i * 0.25 * FS)
    music[start:start + int(0.25 * FS)] += 0.5 * pluck(bass, 0.25) + 0.3 * pluck(lead, 0.25)
for i in range(16):
    start = int(i * 0.125 * FS)
    music[start:start + int(0.05 * FS)] += 0.15 * hat(0.05)
music *= 0.3 / np.max(np.abs(music))

def play(x, label):
    print(f'{label}:')
    display.display(display.Audio(np.clip(x, -1, 1), rate=FS, normalize=False))

## 1. Equalizer
An equalizer boosts or cuts frequency ranges. Three filter shapes cover most needs:
- **Low shelf**: changes everything below a corner frequency by the same amount.
- **Peaking**: changes a band around a center frequency. **Q** sets the width: higher Q, narrower band.
- **High shelf**: changes everything above a corner frequency.

Each is a single biquad from Robert Bristow-Johnson's Audio EQ Cookbook (`audiodsp.rbj_biquad`). The cookbook formulas are pre-warped bilinear transforms, so each filter lands exactly on its frequency (see `filter_design_foundations.ipynb`). Filters in series multiply their responses, so their dB responses add.

In [ ]:
def equalizer(low_f=120, low_gain=0.0, mid_f=1000, mid_gain=0.0, mid_q=1.0, high_f=6000, high_gain=0.0):
    bands = [('low shelf', rbj_biquad('lowshelf', low_f, FS, gain_db=low_gain)),
             ('peaking', rbj_biquad('peaking', mid_f, FS, q=mid_q, gain_db=mid_gain)),
             ('high shelf', rbj_biquad('highshelf', high_f, FS, gain_db=high_gain))]
    sos = np.array([np.concatenate((b, a)) for _, (b, a) in bands])

    freqs = np.geomspace(20, FS / 2, 2000)
    fig, ax = plt.subplots(figsize=(14, 4.5))
    for name, (b, a) in bands:
        _, H = signal.freqz(b, a, worN=freqs, fs=FS)
        ax.semilogx(freqs, 20 * np.log10(np.abs(H)), '--', linewidth=1, label=name)
    _, H = signal.sosfreqz(sos, worN=freqs, fs=FS)
    ax.semilogx(freqs, 20 * np.log10(np.abs(H)), 'k', linewidth=2.5, label='total')
    ax.set_ylim(-15, 15)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('dB')
    ax.set_title('Equalizer response')
    ax.legend(loc='upper left')
    plt.show()

    y = signal.sosfilt(sos, music)
    if np.max(np.abs(y)) > 1:
        print('Output exceeds full scale and is clipped. Lower the boosts.')
    play(music, 'Original')
    play(y, 'Equalized')

display.display(interactive(
    equalizer,
    low_f=widgets.IntSlider(value=120, min=30, max=500, step=10, description='Low (Hz):'),
    low_gain=widgets.FloatSlider(value=0, min=-12, max=12, step=0.5, description='Low (dB):'),
    mid_f=widgets.IntSlider(value=1000, min=200, max=8000, step=50, description='Mid (Hz):'),
    mid_gain=widgets.FloatSlider(value=0, min=-12, max=12, step=0.5, description='Mid (dB):'),
    mid_q=widgets.FloatSlider(value=1.0, min=0.3, max=10, step=0.1, description='Mid Q:'),
    high_f=widgets.IntSlider(value=6000, min=1000, max=16000, step=100, description='High (Hz):'),
    high_gain=widgets.FloatSlider(value=0, min=-12, max=12, step=0.5, description='High (dB):'),
))

## 2. Convolution reverb
A room is an LTI system, so its effect on any sound is described completely by its **room impulse response** (RIR): what a microphone records when a single click is played. Convolving a dry recording with an RIR places it in that room. Real RIRs are measured, usually with a sine sweep. Here one is synthesized from three parts:
- **Direct sound**: a single impulse at time 0.
- **Early reflections**: a few discrete echoes in the first 50 ms, from the nearest walls.
- **Diffuse tail**: random noise with an exponentially decaying envelope, the dense mass of later reflections.

The **reverb time** RT60 is how long the sound takes to decay by 60 dB. A tail envelope of $e^{-t \ln(1000)/\mathrm{RT60}}$ decays by exactly 60 dB in RT60 seconds.

RT60 is measured from an impulse response by **Schroeder backward integration**: the energy remaining after time $t$, $\mathrm{EDC}(t) = \sum_{\tau \ge t} h[\tau]^2$, in dB. For an exponential decay it is a straight line. Its slope between -5 and -25 dB, extrapolated to -60 dB, gives the reverb time (this is the standard "T20" measurement). Measuring the synthetic RIR checks that it has the reverb time it was built with.

A 2 s RIR is 88,200 samples. Direct convolution with 2 s of music would need about $88200^2 \approx 7.8 \times 10^9$ multiplications. `scipy.signal.fftconvolve` does it with FFTs in milliseconds. Real-time convolution reverbs do the same block by block with overlap-add (`scipy.signal.oaconvolve`, and `04_Real_time_Audio_Processing`).

In [ ]:
def synthetic_rir(rt60, fs=FS, direct_to_reverb_db=-6.0):
    t = time_axis(1.5 * rt60, fs)
    envelope = np.exp(-t * np.log(1000) / rt60)
    tail = rng.normal(0, 1, len(t)) * envelope
    tail[:int(0.005 * fs)] = 0                          # the tail starts after the first reflections
    tail *= np.sqrt(10 ** (-direct_to_reverb_db / 10) / np.sum(tail**2))
    h = tail
    h[0] += 1.0                                         # direct sound
    for delay_ms in rng.uniform(5, 50, 6):              # early reflections
        i = int(delay_ms / 1000 * fs)
        h[i] += rng.choice([-1, 1]) * 0.6 * envelope[i]
    return h

def measure_rt60(h, fs=FS):
    """T20: fit the Schroeder decay curve between -5 and -25 dB, extrapolate to -60 dB."""
    edc = np.cumsum(h[::-1]**2)[::-1]
    edc_db = 10 * np.log10(edc / edc[0])
    fit = (edc_db <= -5) & (edc_db >= -25)
    t = np.arange(len(h)) / fs
    slope, intercept = np.polyfit(t[fit], edc_db[fit], 1)
    return -60 / slope, t, edc_db, (slope, intercept)

def reverb(rt60=1.2, mix=0.35):
    h = synthetic_rir(rt60)
    measured, t_h, edc_db, (slope, intercept) = measure_rt60(h)
    wet = signal.fftconvolve(music, h)
    dry = np.concatenate((music, np.zeros(len(wet) - len(music))))
    wet *= np.sqrt(np.mean(dry**2) / np.mean(wet**2))       # match loudness before mixing
    y = (1 - mix) * dry + mix * wet

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    axes[0].plot(t_h, h, linewidth=0.5)
    axes[0].set_title('Room impulse response')
    axes[0].set_xlabel('Time (s)')
    axes[1].plot(t_h, edc_db, label='Schroeder decay curve')
    axes[1].plot(t_h, slope * t_h + intercept, 'r--', label=f'T20 fit: RT60 = {measured:.2f} s')
    axes[1].set_ylim(-80, 5)
    axes[1].set_title(f'Energy decay (target RT60 = {rt60:.2f} s)')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('dB')
    axes[1].legend()
    axes[2].plot(np.arange(len(y)) / FS, y, linewidth=0.5, label='wet/dry mix')
    axes[2].plot(np.arange(len(dry)) / FS, dry, linewidth=0.5, alpha=0.7, label='dry')
    axes[2].set_title(f'Output, mix = {mix:.2f}')
    axes[2].set_xlabel('Time (s)')
    axes[2].legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    play(dry, 'Dry')
    play(y, 'With reverb')

display.display(interactive(
    reverb,
    rt60=widgets.FloatSlider(value=1.2, min=0.2, max=4.0, step=0.1, description='RT60 (s):'),
    mix=widgets.FloatSlider(value=0.35, min=0.0, max=1.0, step=0.05, description='Wet mix:'),
))

## 3. Saturation and oversampling
Saturation passes the signal through a curve that flattens out at high levels, such as $y = \tanh(d\,x)$ with drive $d$. It is the digital version of an overdriven tube or tape. The curve is applied to each sample on its own, so the effect is time-invariant but not linear (`01_Discrete_Time_Signals_and_Systems` section 5). A nonlinear system creates new frequencies: a sine at $f_0$ through tanh comes out with harmonics at $3f_0, 5f_0, 7f_0, \dots$ (odd harmonics only, because tanh is symmetric).

Harmonics above the Nyquist frequency **alias**, landing at $|k f_0 - m f_s|$. These are not harmonics of the note, so they sound harsh and out of tune.

**Oversampling** reduces this: upsample by $L$, saturate at $L f_s$ where many more harmonics fit below the new Nyquist frequency, then low-pass filter and downsample back. Two things limit it:
- Harmonics above $L f_s / 2$ still alias at the high rate, and those that fold back below $f_s/2$ survive. They are weaker, because harmonic levels fall with harmonic number, so this improves quickly as $L$ grows.
- Before downsampling, the filter must remove everything between $f_s/2$ and $L f_s/2$, or it aliases at that step instead. SciPy's default `resample_poly` filter has its cutoff right at $f_s/2$, so harmonics just above 22.05 kHz partly pass. For a 5 kHz tone, the 25 kHz harmonic then folds to 19.1 kHz at about -57 dB, however much you oversample. The sharp filter below is flat to 20 kHz and 100 dB down by 22.05 kHz.

The measurement uses whole seconds and integer frequencies, so every harmonic and every alias falls exactly on an FFT bin. It skips the first and last 0.1 s, where the resampling filters are still starting up and stopping.

In [ ]:
def downsampling_filter(oversample, fs=FS, passband=20000, atten_db=100):
    """Kaiser FIR at oversample * fs: flat to `passband`, at least atten_db down from fs / 2."""
    numtaps, beta = signal.kaiserord(atten_db, (fs / 2 - passband) / (oversample * fs / 2))
    return signal.firwin(numtaps | 1, (passband + fs / 2) / 2, window=('kaiser', beta), fs=oversample * fs)

def saturate(x, drive, oversample=1, sharp_filter=True):
    if oversample == 1:
        return np.tanh(drive * x)
    up = signal.resample_poly(x, oversample, 1)
    window = downsampling_filter(oversample) if sharp_filter else ('kaiser', 5.0)   # ('kaiser', 5.0) is SciPy's default
    return signal.resample_poly(np.tanh(drive * up), 1, oversample, window=window)[:len(x)]

def harmonic_and_alias_levels(f0, drive, oversample, sharp_filter=True, fs=FS):
    """Spectrum in dB (re full-scale sine) of a 1 s steady-state segment, and the harmonic bin mask."""
    x = np.sin(2 * np.pi * f0 * time_axis(1.2, fs))
    y = saturate(x, drive, oversample, sharp_filter)[int(0.1 * fs):int(1.1 * fs)]   # exactly 1 s: 1 Hz bins
    mag_db = 20 * np.log10(np.abs(np.fft.rfft(y)) / (len(y) / 2) + 1e-12)
    freqs = np.arange(len(mag_db))
    is_harmonic = (freqs % f0 == 0) & (freqs // f0 % 2 == 1)
    return freqs, mag_db, is_harmonic

def largest_alias(f0, drive, oversample, sharp_filter=True):
    freqs, mag_db, is_harmonic = harmonic_and_alias_levels(f0, drive, oversample, sharp_filter)
    alias = np.where(is_harmonic, -np.inf, mag_db)
    return f'{alias.max():6.1f} dB at {freqs[np.argmax(alias)]:5d} Hz'

print('5 kHz sine, drive 2.5: largest aliased component')
print(f'{"":>16}{"default filter":>24}{"sharp filter":>24}')
for oversample in [1, 2, 4, 8, 16]:
    print(f'{oversample:>2}x oversampling{largest_alias(5000, 2.5, oversample, False):>24}{largest_alias(5000, 2.5, oversample):>24}')

In [ ]:
def plot_saturation(f0=5000, drive=2.5, oversample=1):
    freqs, mag_db, is_harmonic = harmonic_and_alias_levels(f0, drive, oversample)
    alias = np.where(is_harmonic, -np.inf, mag_db)
    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.plot(freqs, mag_db, color='0.6', linewidth=0.8)
    ax.plot(freqs[is_harmonic], mag_db[is_harmonic], 'go', label='harmonics of f0')
    loud_alias = alias > -100
    ax.plot(freqs[loud_alias], mag_db[loud_alias], 'rx', label='aliases (above -100 dB)')
    ax.set_ylim(-140, 5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('dB')
    ax.set_title(f'tanh({drive} x) of a {f0} Hz sine, {oversample}x oversampling: largest alias {alias.max():.1f} dB')
    ax.legend(loc='upper right')
    plt.show()

display.display(interactive(
    plot_saturation,
    f0=widgets.IntSlider(value=5000, min=100, max=10000, step=100, description='f0 (Hz):'),
    drive=widgets.FloatSlider(value=2.5, min=1.0, max=10.0, step=0.5, description='Drive:'),
    oversample=widgets.Dropdown(options=[1, 2, 4, 8, 16], value=1, description='Oversample:'),
))

### Hearing aliasing
A sine sweep from 100 Hz to 10 kHz, saturated without and with 8x oversampling. Without oversampling, aliased tones sweep *downwards* while the note sweeps up, and are clearly audible in the upper half of the sweep.

In [ ]:
sweep = 0.8 * signal.chirp(time_axis(4.0), 100, 4.0, 10000)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), sharey=True)
for ax, oversample in zip(axes, [1, 8]):
    y = saturate(sweep, 4.0, oversample)
    f, tt, Sxx = signal.spectrogram(y, FS, nperseg=2048)
    S = 10 * np.log10(Sxx + 1e-15)
    ax.pcolormesh(tt, f / 1000, S, shading='auto', vmin=S.max() - 90)
    ax.set_title(f'tanh(4x) of a sweep, {oversample}x oversampling')
    ax.set_xlabel('Time (s)')
    play(y, f'{oversample}x oversampling')
axes[0].set_ylabel('Frequency (kHz)')
plt.tight_layout()
plt.show()